# 4 * 3 Grid World Problem

In [2]:
import numpy as np

def value_iteration(grid, terminal_states, actions, transition_prob, rewards, gamma=0.9, theta=1e-6):
    """
    Parameters:
        grid (tuple): Dimensions of the grid (rows, cols).
        terminal_states (dict): Terminal states with their respective rewards.
        actions (list): Possible actions ['Up', 'Down', 'Left', 'Right'].
        transition_prob (dict): Transition probabilities for actions.
        rewards (float): Reward for non-terminal states.
        gamma (float): Discount factor.
        theta (float): Convergence threshold.

    """
    rows, cols = grid
    V = np.zeros((rows, cols))

    # Initialize terminal states
    for (x, y), reward in terminal_states.items():
        V[x, y] = reward

    while True:
        delta = 0
        new_V = V.copy()

        for i in range(rows):
            for j in range(cols):
                if (i, j) in terminal_states:
                    continue

                action_values = []
                for action in actions:
                    value = 0
                    for outcome, prob in transition_prob[action]:
                        next_i, next_j = i + outcome[0], j + outcome[1]

                        # Stay in place if bumping into a wall
                        if next_i < 0 or next_i >= rows or next_j < 0 or next_j >= cols:
                            next_i, next_j = i, j

                        value += prob * V[next_i, next_j]
                    action_values.append(value)

                new_V[i, j] = rewards + gamma * max(action_values)
                delta = max(delta, abs(new_V[i, j] - V[i, j]))

        V = new_V

        if delta < theta:
            break

    return V

def main():
    # Grid dimensions
    grid = (4, 3)

    # Terminal states with rewards
    terminal_states = {
        (0, 2): +1,  # Adjusted for zero-based indexing
        (1, 2): -1   # Adjusted for zero-based indexing
    }

    # Actions and transition probabilities
    actions = ['Up', 'Down', 'Left', 'Right']
    transition_prob = {
        'Up': [((-1, 0), 0.8), ((0, -1), 0.1), ((0, 1), 0.1)],
        'Down': [((1, 0), 0.8), ((0, -1), 0.1), ((0, 1), 0.1)],
        'Left': [((0, -1), 0.8), ((-1, 0), 0.1), ((1, 0), 0.1)],
        'Right': [((0, 1), 0.8), ((-1, 0), 0.1), ((1, 0), 0.1)]
    }

    # Test for different rewards
    reward_values = [-2, 0.1, 0.02, 1]

    for rewards in reward_values:
        print(f"Value function for r(s) = {rewards}")
        V = value_iteration(grid, terminal_states, actions, transition_prob, rewards)
        print(V)
        print("\n")

if __name__ == "__main__":
    main()

Value function for r(s) = -2
[[-4.10967329 -1.73885616  1.        ]
 [-5.42029205 -3.35954562 -1.        ]
 [-7.01720738 -5.36720648 -3.51983359]
 [-8.45577308 -7.1373839  -5.6886205 ]]


Value function for r(s) = 0.1
[[ 0.99999994  0.99999997  1.        ]
 [ 0.99999989  0.99999982 -1.        ]
 [ 0.99999979  0.99999965  0.99999861]
 [ 0.99999962  0.99999944  0.99999901]]


Value function for r(s) = 0.02
[[ 0.78773964  0.87896037  1.        ]
 [ 0.71101809  0.66504388 -1.        ]
 [ 0.64388729  0.60004987  0.48076206]
 [ 0.58584231  0.55019765  0.50484665]]


Value function for r(s) = 1
[[ 9.99999145  9.99999145  1.        ]
 [ 9.99999145  9.99999145 -1.        ]
 [ 9.99999145  9.99999145  9.99999145]
 [ 9.99999145  9.99999145  9.99999145]]




# **Gbike Bicycle Rental**

In [ ]:
import numpy as np
from scipy.stats import poisson

In [ ]:
# Define constants
MAX_BIKES = 10
MAX_MOVE = 5
RENTAL_REWARD = 10
TRANSFER_COST = 2
PARKING_COST = 4
DISCOUNT = 0.9
REQUEST_RATE = [3, 4]
RETURN_RATE = [3, 2]

In [ ]:
# Poisson distribution probabilities
def poisson_prob(n, lam):
    return poisson.pmf(n, lam)

In [ ]:
# Calculate expected rewards
def expected_rewards_and_values(state, action, value):
    s1, s2 = state
    # Apply the action
    s1 -= action
    s2 += action

    # Cost of action
    cost = abs(action) * TRANSFER_COST
    if action > 0:
        cost -= TRANSFER_COST  # First bike is free when moving from 1 to 2

    # Initialize reward and value
    reward = -cost
    value_sum = 0.0

    # Iterate over possible requests and returns
    for req1 in range(0, MAX_BIKES + 1):
        for req2 in range(0, MAX_BIKES + 1):
            for ret1 in range(0, MAX_BIKES + 1):
                for ret2 in range(0, MAX_BIKES + 1):
                    # Probabilities
                    prob = (poisson_prob(req1, REQUEST_RATE[0]) *
                            poisson_prob(req2, REQUEST_RATE[1]) *
                            poisson_prob(ret1, RETURN_RATE[0]) *
                            poisson_prob(ret2, RETURN_RATE[1]))

                    # Number of rentals
                    rented1 = min(req1, s1)
                    rented2 = min(req2, s2)

                    # Calculate reward for rentals
                    immediate_reward = RENTAL_REWARD * (rented1 + rented2)

                    # State after rentals and returns
                    new_s1 = min(s1 - rented1 + ret1, MAX_BIKES)
                    new_s2 = min(s2 - rented2 + ret2, MAX_BIKES)

                    # Parking penalty
                    parking_penalty = 0
                    if new_s1 > 10:
                        parking_penalty += PARKING_COST
                    if new_s2 > 10:
                        parking_penalty += PARKING_COST

                    # Total reward
                    immediate_reward -= parking_penalty

                    # Value contribution
                    value_sum += prob * (immediate_reward + DISCOUNT * value[new_s1, new_s2])

    return reward, value_sum

In [ ]:
# Policy evaluation
def policy_evaluation(policy, value, theta=0.1):
    while True:
        delta = 0
        new_value = value.copy()
        for s1 in range(MAX_BIKES + 1):
            for s2 in range(MAX_BIKES + 1):
                action = policy[s1, s2]
                reward, value_sum = expected_rewards_and_values((s1, s2), action, value)
                new_value[s1, s2] = reward + value_sum
                delta = max(delta, abs(new_value[s1, s2] - value[s1, s2]))
        value = new_value
        if delta < theta:
            break
    return value

In [ ]:
# Policy improvement
def policy_improvement(value, policy):
    policy_stable = True
    for s1 in range(MAX_BIKES + 1):
        for s2 in range(MAX_BIKES + 1):
            old_action = policy[s1, s2]
            action_values = []
            for action in range(-MAX_MOVE, MAX_MOVE + 1):
                if (0 <= s1 - action <= MAX_BIKES) and (0 <= s2 + action <= MAX_BIKES):
                    reward, value_sum = expected_rewards_and_values((s1, s2), action, value)
                    action_values.append((reward + value_sum, action))
            _, best_action = max(action_values)
            policy[s1, s2] = best_action
            if best_action != old_action:
                policy_stable = False
    return policy, policy_stable

In [ ]:
# Main policy iteration function
def policy_iteration():
    policy = np.zeros((MAX_BIKES + 1, MAX_BIKES + 1), dtype=int)
    value = np.zeros((MAX_BIKES + 1, MAX_BIKES + 1))
    iterations = 0

    while True:
        iterations += 1
        value = policy_evaluation(policy, value)
        policy, policy_stable = policy_improvement(value, policy)
        if policy_stable:
            break

    print(f"Policy iteration converged after {iterations} iterations.")
    return policy, value


In [ ]:
# Solve the problem
policy, value = policy_iteration()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_policy(policy):
    plt.figure(figsize=(10, 8))
    plt.imshow(policy, cmap="coolwarm", origin="lower")
    plt.colorbar(label="Action (Bike Transfer)")
    plt.title("Policy (Bike Transfer Actions)")
    plt.xlabel("Bikes at Location 2")
    plt.ylabel("Bikes at Location 1")
    plt.xticks(range(MAX_BIKES + 1))
    plt.yticks(range(MAX_BIKES + 1))
    plt.grid(visible=True, which='both', linestyle='--', linewidth=0.5)
    plt.show()

def plot_value_function(value):
    plt.figure(figsize=(10, 8))
    plt.imshow(value, cmap="viridis", origin="lower")
    plt.colorbar(label="Value")
    plt.title("Value Function")
    plt.xlabel("Bikes at Location 2")
    plt.ylabel("Bikes at Location 1")
    plt.xticks(range(MAX_BIKES + 1))
    plt.yticks(range(MAX_BIKES + 1))
    plt.grid(visible=True, which='both', linestyle='--', linewidth=0.5)
    plt.show()

plot_policy(policy)
plot_value_function(value)